# ***IMPORTS***

In [1]:
#pip install openpyxl
#pip install beautifulsoup4

import openpyxl

from bs4 import BeautifulSoup
import requests
import os
from datetime import datetime
import time
import winsound
#import pandas as pd


import warnings
warnings.filterwarnings('ignore')

## **FUNCIONES Y VARIABLES GLOBALES**

In [2]:
# Conectar y obtener datos desde Zooplus

#Se obtiene el User Agent desde http://httpbin.org/get
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36", "Accept-Encoding":"gzip, deflate", "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8", "DNT":"1","Connection":"close", "Upgrade-Insecure-Requests":"1"}

#Funcion para obtener el precio de un producto
def get_zooplus_price(web_prefix, ean):
    url = web_prefix + ean    
    #Se obtiene el HTML del producto
    page = requests.get(url, headers=headers, verify=False)
    soup1 = BeautifulSoup(page.content, "html.parser")

    #Se extrae el precio del HTML
    try:
        price = soup1.find("span", { "class" : "z-price__amount z-price__amount--standard" }).text.strip()
       
    except:
        ean12 = str(ean).zfill(12)
        url = web_prefix + ean12

        page = requests.get(url, headers=headers, verify=False)
        soup1 = BeautifulSoup(page.content, "html.parser")
        
        try:
            price = soup1.find("span", { "class" : "z-price__amount z-price__amount--standard" }).text.strip()
            
        except:
            ean13 = str(ean).zfill(13)
            url = web_prefix + ean13

            page = requests.get(url, headers=headers, verify=False)
            soup1 = BeautifulSoup(page.content, "html.parser")
            
            try:
                price = soup1.find("span", { "class" : "z-price__amount z-price__amount--standard" }).text.strip()
            
            except:
                url = web_prefix + ean    
    
                page = requests.get(url, headers=headers, verify=False)
                soup1 = BeautifulSoup(page.content, "html.parser")
                
                try:
                    price = soup1.find("span", {"class" : "z-price__amount"}).text.strip() 
                except:
                    ean12 = str(ean).zfill(12)
                    url = web_prefix + ean12

                    page = requests.get(url, headers=headers, verify=False)
                    soup1 = BeautifulSoup(page.content, "html.parser")
                    try:
                        price = soup1.find("span", {"class" : "z-price__amount"}).text.strip()
                    except:
                        ean13 = str(ean).zfill(13)
                        url = web_prefix + ean13

                        page = requests.get(url, headers=headers, verify=False)
                        soup1 = BeautifulSoup(page.content, "html.parser")
                    try:
                        price = soup1.find("span", {"class" : "z-price__amount"}).text.strip()
                    except AttributeError:
                        price = "No se ha podido obtener el precio para este producto."
        
    return price

#funciones para hora y fecha

def actual_date():
    now = datetime.now()
    date_now = now.date()
   
    return date_now

def actual_hour():
    now = datetime.now()
    hour_now = now.time()

    return hour_now

def pitido():
    frecuency = 1000
    duration = 100
    winsound.Beep(frecuency, duration)

## **CONFIGURACIÓN DE ARCHIVO DE EXCEL**

In [3]:
#Incluir la localizacion del fichero excel con los datos de Zooplus:
directory = os.getcwd()
path = directory + "\\Precios.xlsx"
#print(directory)

# Se abre el workbook y la hoja seleccionados (donde se guardo el documento por ultima vez)
wb_obj = openpyxl.load_workbook(path)
sheet_obj = wb_obj.active

# Obtener el numero de filas y columnas usadas
row_limit = sheet_obj.max_row
column_limit = sheet_obj.max_column
  
print("Total de filas:", row_limit)
print("Total de columnas:", column_limit)

#Introducir el numero de columna en la que se encuentra el EAN (A=1, B=2, C=3...)
ean_col = 3

#Introducir el numero de la fila en la que se encuentra el primer valor de EAN
first_ean_row = 2

#Introducir el numero de columna en la que se encuentra el precio
precio_col = 4

Total de filas: 4218
Total de columnas: 4


## **EJECUTAR TAREA DE RELLENADO DE PRECIOS**

In [4]:
# Cerrar el fichero Excel antes de ejecutar esta celda

#Introducir el formato de la web que se va a buscar
web_prefix = "https://www.zooplus.es/search/results?q="

date = actual_date()
hour = actual_hour()
time1 = time.time()

print(("INICIADO EL {} A LAS {} HORAS").format(date, hour))

for i in range(first_ean_row, row_limit + 1): 
    cell_obj = sheet_obj.cell(row = i, column = ean_col)
    ean = str(cell_obj.value)
    
    if ean == None:
        ean = ""
    
    
    #url = web_prefix#+ean
    try:
        price = get_zooplus_price(web_prefix, ean)
        print(("{}: {} tiene un precio de: {}").format(str(i-1).zfill(4), ean, price))
    
        #Se introduce el precio en el Excel
        c1 = sheet_obj.cell(row = i, column = precio_col)
        c1.value = price
    except:
        print("Hemos tenido un error de conexión, voy a descansar 10 segundos. ZZZZZZzzzzz...")
        time.sleep(10)
        print("Buena siesta!!! Vuelvo a la carga!!!!")
        try:
            price = get_zooplus_price(web_prefix, ean)
            print(("{}: {} tiene un precio de: {}").format(str(i-1).zfill(4), ean, price))
    
            #Se introduce el precio en el Excel
            c1 = sheet_obj.cell(row = i, column = precio_col)
            c1.value = price
        except:
            break
    
#Se guarda el fichero Excel con los cambios aplicados
wb_obj.save(path)
time2 = time.time()
hour1 = actual_hour()
dif_time = time2 - time1
print(("FINALIZADO EL {} A LAS {} HORAS").format(date, hour1))
print(("REALIZADO EN: {} horas").format((dif_time/60)/60))
print("ARCHIVO GUARDADO / PROGRAMA FINALIZADO")
print('\a')
pitido()

INICIADO EL 2024-03-04 A LAS 09:30:50.028056 HORAS
0001: 4062911008618 tiene un precio de: 19,49 €
0002: 4062911004481 tiene un precio de: 13,99 €
0003: 4260077045496 tiene un precio de: 17,99 €
0004: 052742047447 tiene un precio de: 35,99 €
0005: 5000161033072 tiene un precio de: 3,29 €
0006: 052742025209 tiene un precio de: 16,29 €
0007: 4008239610393 tiene un precio de: 13,29 €
0008: 4260358513584 tiene un precio de: 1,99 €
0009: 4001967156010 tiene un precio de: No se ha podido obtener el precio para este producto.
0010: 4062911021891 tiene un precio de: 18,99 €
0011: 052742211107 tiene un precio de: 13,19 €
0012: 4054651005422 tiene un precio de: 2,79 €
0013: 4021158716908 tiene un precio de: 59,49 €
0014: 4004218726338 tiene un precio de: 12,99 €
0015: 4017721838801 tiene un precio de: 4,29 €
0016: 8437013576192 tiene un precio de: 2,69 €
0017: 3065890138254 tiene un precio de: 10,99 €
0018: 4062911000636 tiene un precio de: 6,99 €
0019: 4260535818112 tiene un precio de: 12,99 €
